In [4]:
import os, sys

# --- Force HF cache to /tmp (NOT Drive) ---
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets_cache"

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = "/content/drive/MyDrive/HateSpeech_NLP"

REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    os.system(f"cd {REPO_PATH} && git pull origin main")

os.system(f"pip install -q -e {REPO_PATH}")

if f"{REPO_PATH}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_PATH}/src")

os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("✅ Environment ready. REPO_PATH:", REPO_PATH)
print("✅ DRIVE_ROOT:", DRIVE_ROOT)

Mounted at /content/drive
✅ Environment ready. REPO_PATH: /content/hate-speech-detection
✅ DRIVE_ROOT: /content/drive/MyDrive/HateSpeech_NLP


In [5]:
import os
import subprocess
import threading
import time
import sys

# Set model source
os.environ["HF_MODEL_ID"] = "thong7d/vihsd-xlmr-hate-speech"

# Install ngrok
os.system("pip install -q pyngrok")
from pyngrok import ngrok

# Cấu hình Authtoken
ngrok.set_auth_token("3CO8IpvERS1b7tqBq7BC9lygnyP_3b93zW7mEhaJWWCycRUEo")

# Khai báo biến REPO_PATH trỏ về thư mục làm việc hiện tại (hoặc thay bằng đường dẫn cụ thể)
REPO_PATH = os.getcwd()

# Start FastAPI in background thread
def run_uvicorn():
    import uvicorn
    # Import app to trigger model loading
    sys.path.insert(0, f"{REPO_PATH}/src")
    from api.app import app
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(10)  # Wait for model to load

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f"✅ FastAPI is live!")
print(f"   Public URL:  {public_url}")
print(f"   Docs:        {public_url}/docs")
print(f"   Health:      {public_url}/health")

✅ FastAPI is live!
   Public URL:  NgrokTunnel: "https://uncork-bubbly-stiffly.ngrok-free.dev" -> "http://localhost:8000"
   Docs:        NgrokTunnel: "https://uncork-bubbly-stiffly.ngrok-free.dev" -> "http://localhost:8000"/docs
   Health:      NgrokTunnel: "https://uncork-bubbly-stiffly.ngrok-free.dev" -> "http://localhost:8000"/health


In [9]:
import requests

# Điền đúng Public URL của ngrok (bắt buộc có https://)
API_URL = "https://uncork-bubbly-stiffly.ngrok-free.dev"

test_cases = [
    "Bài viết rất hay, cảm ơn tác giả!",
    "Đồ ngu, mày biết gì mà nói",
    "Loại người như mày không xứng đáng sống",
]

print("📊 API Test Results:")
print("=" * 70)
for text in test_cases:
    response = requests.post(f"{API_URL}/predict", json={"text": text})
    result = response.json()
    print(f"Text:       {result['text'][:60]}...")
    print(f"Label:      {result['label']} (conf: {result['confidence']:.3f})")
    print(f"Scores:     {result['scores']}")
    print(f"Borderline: {result['is_borderline']}")
    print(f"Latency:    {result['latency_ms']:.1f}ms")
    print("-" * 70)

📊 API Test Results:
Loading model from: thong7d/vihsd-xlmr-hate-speech


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded on cuda
INFO:     136.117.162.99:0 - "POST /predict HTTP/1.1" 200 OK
Text:       Bài viết rất hay, cảm ơn tác giả!...
Label:      CLEAN (conf: 1.000)
Scores:     {'CLEAN': 0.9996, 'OFFENSIVE': 0.0001, 'HATE': 0.0003}
Borderline: False
Latency:    769.3ms
----------------------------------------------------------------------
INFO:     136.117.162.99:0 - "POST /predict HTTP/1.1" 200 OK
Text:       Đồ ngu, mày biết gì mà nói...
Label:      HATE (conf: 0.997)
Scores:     {'CLEAN': 0.0019, 'OFFENSIVE': 0.0011, 'HATE': 0.9971}
Borderline: False
Latency:    18.4ms
----------------------------------------------------------------------
INFO:     136.117.162.99:0 - "POST /predict HTTP/1.1" 200 OK
Text:       Loại người như mày không xứng đáng sống...
Label:      CLEAN (conf: 0.636)
Scores:     {'CLEAN': 0.636, 'OFFENSIVE': 0.001, 'HATE': 0.363}
Borderline: True
Latency:    18.1ms
----------------------------------------------------------------------


In [10]:
import gradio as gr
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HF_MODEL_ID = "thong7d/vihsd-xlmr-hate-speech"  # ← Update
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_ID)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

LABEL_MAP = {0: "CLEAN ✅", 1: "OFFENSIVE ⚠️", 2: "HATE 🚫"}
LABEL_COLORS = {0: "#4CAF50", 1: "#FF9800", 2: "#F44336"}

def predict_text(text):
    """Predict hate speech label for input text."""
    if not text or len(text.strip()) < 3:
        return "Please enter at least 3 characters.", "", ""

    encoding = tokenizer(
        text, max_length=128, padding="max_length",
        truncation=True, return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred_id = int(probs.argmax())
    confidence = float(probs.max())

    label_text = LABEL_MAP[pred_id]
    scores_text = "\n".join([
        f"CLEAN:     {probs[0]:.4f}",
        f"OFFENSIVE: {probs[1]:.4f}",
        f"HATE:      {probs[2]:.4f}",
    ])
    conf_text = f"{confidence:.2%}"

    return label_text, conf_text, scores_text

demo = gr.Interface(
    fn=predict_text,
    inputs=gr.Textbox(
        label="Enter Vietnamese text",
        placeholder="Nhập văn bản tiếng Việt tại đây...",
        lines=3,
    ),
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Textbox(label="Confidence"),
        gr.Textbox(label="All Scores"),
    ],
    title="🛡️ Vietnamese Hate Speech Detector",
    description=(
        "Powered by XLM-RoBERTa fine-tuned on the ViHSD dataset (~33K comments). "
        "Classifies text as CLEAN, OFFENSIVE, or HATE."
    ),
    examples=[
        ["Bài viết rất hay, cảm ơn tác giả!"],
        ["Mày ngu quá, biết gì mà nói"],
        ["Loại người như mày không xứng đáng sống trên đời này"],
    ],
    theme=gr.themes.Soft(),
    allow_flagging="never",
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb350044bd688ea424.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
